# Gỡ lỗi vòng train PyTorch hỏng mà không báo lỗi

Notebook này train một ResNet nhỏ trên 10 000 ảnh CIFAR-10 bằng SGD + OneCycle, có gradient accumulation và
augmentation. Nó chạy hết không exception, loss có giảm, nhưng validation accuracy thấp hơn hẳn mức một ResNet
cỡ này phải đạt. Trong các cell chấm điểm có **5 bug im lặng**, mỗi bug một chỗ, đều là lỗi đặc trưng của PyTorch
(khác các bug numpy ở module 01). Đề không nói chúng ở đâu.

**Cách làm bài**
- Tìm và sửa bug ngay trong các cell định nghĩa. Không đổi chữ ký hàm, siêu tham số mặc định của `train`,
  kiến trúc dự định (stem, 3 `ResBlock` w → 2w → 4w, average pool, `Linear`) và `DEVICE`.
- Thêm cell `explore` tuỳ ý để đo. Ghi lại từng bug trong `BUGS.md`.
- Chấm: `uv run pytest tracks/ai-engineer/modules/02-pytorch-deep-dive/exercises/advanced/02-debug-pytorch-training -q`

In [ ]:
# explore: giới hạn luồng CPU, tìm gốc repo, kiểm CIFAR-10 (137 MB, tải một lần bằng env gốc)
import os

os.environ.setdefault("OMP_NUM_THREADS", "4")   # đặt TRƯỚC khi import torch; GPU dùng chung, CPU cũng vậy
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
MODULE = ROOT / "tracks/ai-engineer/modules/02-pytorch-deep-dive"
CIFAR = ROOT / "data/raw/cifar10"
if not (CIFAR.is_dir() and any(CIFAR.iterdir())):
    # kernel ai-gpu không có thư viện tải dữ liệu: khi đó chạy ở terminal `uv run python data/download_dataset.py cifar10`
    sys.path.insert(0, str(ROOT))
    from data.download_dataset import ensure

    ensure("cifar10")
print(CIFAR, sorted(p.name for p in (CIFAR / "plain_text").iterdir()))

In [ ]:
import time

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

DEVICE = "cpu"   # bài này chạy trên CPU cho mọi người; đừng đổi khi đang gỡ lỗi
MEAN = torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1)
STD = torch.tensor([0.2470, 0.2435, 0.2616]).view(1, 3, 1, 1)

## Dữ liệu và model

In [ ]:
def preprocess(images: torch.Tensor) -> torch.Tensor:
    """Ảnh uint8 dạng (N, H, W, C) như đọc từ PIL / Hugging Face datasets -> float (N, C, H, W) đã chuẩn hoá."""
    n, h, w, c = images.shape
    x = images.reshape(n, c, h, w).float() / 255
    return (x - MEAN) / STD

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, c_in: int, c_out: int, stride: int = 1):
        super().__init__()
        self.conv1 = nn.Conv2d(c_in, c_out, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(c_out)
        self.conv2 = nn.Conv2d(c_out, c_out, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(c_out)
        self.skip = nn.Identity() if stride == 1 and c_in == c_out else nn.Sequential(
            nn.Conv2d(c_in, c_out, 1, stride, bias=False), nn.BatchNorm2d(c_out))

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        return F.relu(self.bn2(self.conv2(out)) + self.skip(x))


class Net(nn.Module):
    """ResNet nhỏ cho ảnh 32x32: stem, 3 tầng ResBlock (w, 2w, 4w), global average pool, Linear."""

    def __init__(self, w: int = 16, n_classes: int = 10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, w, 3, 1, 1, bias=False), nn.BatchNorm2d(w), nn.ReLU())
        self.blocks = [ResBlock(w, w), ResBlock(w, 2 * w, 2), ResBlock(2 * w, 4 * w, 2)]
        self.head = nn.Linear(4 * w, n_classes)

    def forward(self, x):
        x = self.stem(x)
        for block in self.blocks:
            x = block(x)
        return self.head(x.mean(dim=(2, 3)))

## Loss, đánh giá, một bước train

In [ ]:
def loss_fn(logits: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """Loss phân loại 10 lớp, trung bình trên batch."""
    probs = F.softmax(logits, dim=1)
    return F.cross_entropy(probs, y)

In [ ]:
def evaluate(model: nn.Module, x: torch.Tensor, y: torch.Tensor, batch_size: int = 500) -> float:
    """Accuracy trên (x đã preprocess, y)."""
    model.eval()
    hits = 0
    with torch.no_grad():
        for i in range(0, len(x), batch_size):
            hits += (model(x[i:i + batch_size]).argmax(1) == y[i:i + batch_size]).sum().item()
    return hits / len(x)

In [ ]:
def train_step(model: nn.Module, x: torch.Tensor, y: torch.Tensor, optimizer: torch.optim.Optimizer,
               accum_steps: int = 2) -> float:
    """Một bước optimizer cho batch (x, y), chia thành accum_steps micro-batch bằng nhau để tiết kiệm bộ nhớ."""
    total = 0.0
    for xb, yb in zip(x.chunk(accum_steps), y.chunk(accum_steps)):
        optimizer.zero_grad()
        loss = loss_fn(model(xb), yb) / accum_steps
        loss.backward()
        total += loss.item()
    optimizer.step()
    return total

## Vòng train

In [ ]:
def train(x_train: torch.Tensor, y_train: torch.Tensor, x_val: torch.Tensor, y_val: torch.Tensor,
          seed: int = 0, epochs: int = 4, batch_size: int = 128, lr: float = 0.2, w: int = 16) -> dict:
    """Train Net(w) bằng SGD Nesterov + OneCycle. x_* là ảnh uint8 (N, H, W, C), y_* nhãn int64.

    Trả {"model": model, "loss": [loss trung bình mỗi epoch], "val_acc": [accuracy sau mỗi epoch]}.
    """
    torch.manual_seed(seed)
    gen = torch.Generator().manual_seed(seed)
    model = Net(w).to(DEVICE)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, nesterov=True, weight_decay=5e-4)
    steps = epochs * (len(x_train) // batch_size)
    sched = torch.optim.lr_scheduler.OneCycleLR(optimizer, lr, total_steps=steps, pct_start=0.25)
    xt, xv = preprocess(x_train), preprocess(x_val)
    history = {"model": model, "loss": [], "val_acc": []}
    print(f"val acc trước khi train: {evaluate(model, xv, y_val):.3f}")
    for epoch in range(epochs):
        order = torch.randperm(len(xt), generator=gen)
        losses = []
        for i in range(len(xt) // batch_size):
            idx = order[i * batch_size:(i + 1) * batch_size]
            xb = xt[idx]
            flip = torch.rand(len(idx), generator=gen) < 0.5          # augmentation: lật ngang
            xb = torch.where(flip.view(-1, 1, 1, 1), xb.flip(3), xb)
            losses.append(train_step(model, xb, y_train[idx], optimizer))
            sched.step()
        history["loss"].append(float(np.mean(losses)))
        history["val_acc"].append(evaluate(model, xv, y_val))
        print(f"epoch {epoch + 1}: loss {history['loss'][-1]:.3f} | val acc {history['val_acc'][-1]:.3f}")
    return history

## Chạy trên CIFAR-10

In [ ]:
# explore: CIFAR-10 thật: 10 000 ảnh train đầu, 2 000 ảnh test đầu làm validation, ở dạng (N, H, W, C) như PIL trả về.
# Decode bằng data.py của bài Basic 03 (của bạn, hoặc lời giải mẫu nếu chưa làm)
import importlib.util

for path in [MODULE / "exercises/basic/03-fast-dataloader/starter/data.py", MODULE / "solutions/basic/03-fast-dataloader/data.py"]:
    spec = importlib.util.spec_from_file_location(f"basic03_{path.parent.name}", path)
    data_mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(data_mod)
    try:
        xs, ys = data_mod.decode_cifar_parquet(CIFAR / "plain_text/train-00000-of-00001.parquet")
        xv, yv = data_mod.decode_cifar_parquet(CIFAR / "plain_text/test-00000-of-00001.parquet")
        break
    except NotImplementedError:
        continue
x_train, y_train = xs[:10_000].permute(0, 2, 3, 1).contiguous(), ys[:10_000]
x_val, y_val = xv[:2_000].permute(0, 2, 3, 1).contiguous(), yv[:2_000]
print("train", tuple(x_train.shape), x_train.dtype, "| val", tuple(x_val.shape), "| nhãn", torch.bincount(y_train).tolist())

In [ ]:
# explore: chạy thử (khoảng 30 giây trên CPU)
start = time.perf_counter()
hist = train(x_train, y_train, x_val, y_val, seed=0)
print(f"{time.perf_counter() - start:.0f} giây | val acc cuối {hist['val_acc'][-1]:.3f}")